# 02 · Stage 1 — ImageEnvironment

**Question:** can an agent that may only call the Environment API find
placements that hurt YOLO?

No physics and no RL yet.  The scene is a `World`: a real cutout of one object
sitting on a plain background.  The agent proposes `(x, y, size, rotation)`
for its own patch object; the environment validates it, renders the scene,
runs the frozen victim, and returns one scalar.

In [ ]:
# Section 1: Setup
import sys, pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "configs" / "default.yaml").exists():
    ROOT = ROOT.parent          # running from notebooks/
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from configs.loader import load_config, build_victim, resolve

cfg = load_config()
print("project root:", ROOT)
print("victim config:", cfg["victim"])

In [ ]:
from configs.loader import build_stage1_env
from environments.sealed import seal

victim = build_victim(cfg)
env = build_stage1_env(cfg, victim)
api = seal(env)                      # what the agent gets
print(api.action_space().describe())

## Section 2: Environment

The full loop lives *inside* `step()`:
`validate -> render -> victim -> evaluate -> reward`.

In [ ]:
obs = api.reset(seed=0)
baseline = env.pop_telemetry()               # experimenter-side telemetry
print("clean detection:", baseline["baseline_class"], f"{baseline['baseline_confidence']:.3f}")

# size is capped at the target's own silhouette -- the patch can never be
# bigger than the object it attacks (docs/DESIGN.md section 5)
tx, ty = cfg["stage1"]["target_center"]
max_size = api.action_space().high[2]
n_texture = api.action_space().n - 4
texture = np.random.default_rng(0).uniform(0, 1, n_texture)  # a random pattern
obs, reward, terminated, truncated, info = api.step(
    np.concatenate([[tx, ty, max_size, 20.0], texture])
)
print("reward:", round(reward, 4), "| info:", info)

## Section 3: Visualization

In [ ]:
from evaluation.plots import draw_detections

clean = env.clean_image()
attacked = env.render_human()
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(draw_detections(clean, env.victim_report(clean), highlight=baseline["baseline_class"]))
axes[0].set_title("clean"); axes[0].axis("off")
axes[1].imshow(draw_detections(attacked, env.victim_report(attacked), highlight=baseline["baseline_class"]))
axes[1].set_title("one hand-picked placement"); axes[1].axis("off")
plt.show()

## Section 4: Baseline — Random and Greedy search

Both baselines see only `action_space()` and the scalar reward.  Raise
`BUDGET` for a stronger (slower) search.

In [ ]:
from agents.random_agent import RandomAgent
from agents.greedy_agent import GreedyAgent
from evaluation.runner import run_episodes
from evaluation.metrics import summarize
from evaluation.report import format_table

BUDGET = 60          # scripts/run_stage1.py uses cfg["stage1"]["search_episodes"]

results, summaries, curves = {}, [], {}
for name, agent in [("random", RandomAgent(seed=0)), ("greedy", GreedyAgent(seed=0))]:
    records, traces = run_episodes(env, agent, BUDGET, method=name, variant="photo", seed=0)
    results[name] = (records, traces)
    curves[name] = [r.best_confidence for r in records]
    s = summarize(records); s["label"] = name
    summaries.append(s)

print(format_table(summaries))

## Section 5: Attack Agent

Stage 1 deliberately stops at search — PPO arrives in Stage 2, where the world
has dynamics worth learning.  The best placement found so far:

In [ ]:
best_method = min(curves, key=lambda k: min(curves[k]))
records, traces = results[best_method]
best_idx = int(np.argmin([r.best_confidence for r in records]))
best_action = np.array(traces[best_idx].telemetry["steps"][0]["action"])
print("best method:", best_method, "| action:", np.round(best_action, 1))

api.reset(seed=0)
api.step(best_action)
best_frame = env.render_human()
best_conf = env.pop_telemetry()["steps"][-1]["current_confidence"]
print(f"confidence {baseline['baseline_confidence']:.3f} -> {best_conf:.3f}")

## Section 6: Evaluation

In [ ]:
import pandas as pd
from evaluation.metrics import to_dataframe

df = pd.concat([to_dataframe(r) for r, _ in results.values()])
df.groupby("method")[
    ["baseline_confidence", "best_confidence", "confidence_drop", "total_reward", "success"]
].mean().round(4)

## Section 7: Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for name, values in curves.items():
    axes[0].plot(np.minimum.accumulate(values), label=name, linewidth=2)
axes[0].axhline(baseline["baseline_confidence"], ls="--", c="k", lw=1, label="clean")
axes[0].set_xlabel("queries to the environment"); axes[0].set_ylabel("best confidence found")
axes[0].grid(alpha=.3); axes[0].legend(); axes[0].set_title("search progress")

axes[1].imshow(draw_detections(best_frame, env.victim_report(best_frame), highlight=baseline["baseline_class"]))
axes[1].axis("off"); axes[1].set_title(f"best attack ({best_method}): {best_conf:.3f}")
plt.show()

### Stage 1 verdict

Stage 1 is done when the attacked confidence is clearly below the clean
confidence — i.e. an agent restricted to the Environment API *can* move the
victim model.  Full run with the configured budget:

```
uv run python scripts/run_stage1.py
```